In [521]:
import baltic as bt
import pandas as pd
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
from datetime import datetime as dt
from datetime import timedelta
import time
#import pymc3
import math
import arviz as az
#from hpd import hpd
import scipy.stats as stats
from io import StringIO
import altair as alt
from altair import datum
alt.data_transformers.disable_max_rows()
import seaborn as sns

from zipfile import ZipFile
import scipy as sp


import sys, subprocess, glob, os, shutil, re, importlib
from subprocess import call
import imp

%matplotlib inline
import matplotlib as mpl
from matplotlib import pyplot as plt
import matplotlib.patheffects as path_effects
import matplotlib.lines as mlines
from matplotlib.font_manager import FontProperties
import matplotlib.colors as clr
from matplotlib import rc
import textwrap as textwrap
from textwrap import wrap

import warnings
warnings.filterwarnings('ignore')

from scipy.special import binom

In [522]:
##reads in all the Nes from BEAST log files
def read_in_Ne_changes_mascot(log_file_path):
    
    Ne_skyline_dict = {"sample":[]}
    
    with open(log_file_path, "r") as infile:
        line_number = 0
        for line in infile:
            line_number += 1
            if not line.startswith("#"):  # log combiner will sometimes put the entire xml at the start of the log file
                # use the first line to find the migration rate columns
                #print(line)
            # use the first line to find the migration rate columns
                if "posterior" in line:
                    all_cols = line.split("\t")
                    Ne_column_indices = []   # list to store column indices
                    Nes_key = {}   # dictionary to store the column index to map to column name

                    for i in range(len(all_cols)):
                        col = all_cols[i]
                        if "Ne." in col:
                            Ne_column_indices.append(i)

                    # make an empty dictionary to store Nes and generate dictionary to convert index to name
                    for n in Ne_column_indices:
                        name = line.split("\t")[n]
                        #deme = name.split(".")[1]# the syntax here is "Ne_region.1" where region is deme and 1 is interval 1
                        interval = name.split(".")[1]
                       
                        Nes_key[n] = name
                        Ne_skyline_dict[name] = []


                # read in actual parameter estimates and store in dictionary
                else:
                    sample = line.split("\t")[0]
                    Ne_skyline_dict["sample"].append(sample)

                    for index in Ne_column_indices:
                        name = Nes_key[index]
                        Ne_skyline_dict[name].append(line.split("\t")[index])
                    
                
    return(Ne_skyline_dict)


In [523]:
# make a new dataframe that summarizes the 95% HPD estimate with mean for each deme and interval 
def generate_summary_df(input_df):
    
    
    new_df = pd.DataFrame()

    for i in input_df.columns.tolist():
        if "Ne." in i:
            #deme = i.split(".")[1]
           # print(deme)
            #interval = 
            #print(interval)
#             if "\n" in i.split(".")[2]:
#                 interval = i.split(".")[2][0:2]
#             else:
            interval = i.split(".")[1]
           # print(interval)
            #print(interval)
            #print(i)
            #next_interval = int(interval)+1
            local_series = input_df[i].astype('float').to_numpy()
            #print(local_series)
            mean_log = local_series.mean()
            median_log = np.median(local_series)
            mean_linear = np.mean(np.exp(local_series))
            median_linear = np.median(np.exp(local_series))
            hpd_95 = az.hdi(local_series, 0.95)
            lower_hpd_log_95 = hpd_95[0]
            lower_hpd_linear_95 = math.exp(lower_hpd_log_95)
            upper_hpd_log_95 = hpd_95[1]
            upper_hpd_linear_95 = math.exp(upper_hpd_log_95)
            hpd_50 = az.hdi(local_series, 0.50)
            lower_hpd_log_50 = hpd_50[0]
            lower_hpd_linear_50 = math.exp(lower_hpd_log_50)
            upper_hpd_log_50 = hpd_50[1]
            upper_hpd_linear_50 = math.exp(upper_hpd_log_50)
            
#             try:
#                 next_local_series = input_df["SkylineNe"+"."+ str(deme) +"." + str(next_interval)].astype('float').to_numpy()
#                 diff_series = np.subtract(local_series, next_local_series)
#                 #print(local_series)
#                 #print(next_local_series)
#                 #print(diff_series)
#                 diff_mean_log = diff_series.mean()
#                 diff_median_log = np.median(diff_series)
#                 diff_hpd_95 = az.hdi(diff_series, 0.95)
#                 diff_lower_hpd_log_95 = diff_hpd_95[0]
#                 diff_lower_hpd_linear_95 = math.exp(diff_lower_hpd_log_95)
#                 diff_upper_hpd_log_95 = diff_hpd_95[1]
#                 diff_upper_hpd_linear_95 = math.exp(diff_upper_hpd_log_95)
#                 diff_hpd_50 = az.hdi(diff_series, 0.50)
#                 diff_lower_hpd_log_50 = diff_hpd_50[0]
#                 diff_lower_hpd_linear_50 = math.exp(diff_lower_hpd_log_50)
#                 diff_upper_hpd_log_50 = diff_hpd_50[1]
#                 diff_upper_hpd_linear_50 = math.exp(diff_upper_hpd_log_50)
#             except KeyError:
#                 pass   
            
            try:
                local_df = pd.DataFrame.from_dict({"interval":interval, "mean_Ne_log":mean_log,"mean_Ne_linear":mean_linear, 
                                                   "median_Ne_log" : median_log, "median_Ne_linear": median_linear, 
                                                   "upper_hpd_log_95":upper_hpd_log_95,"lower_hpd_log_95":[lower_hpd_log_95], 
                                                   "upper_hpd_log_50":upper_hpd_log_50,"lower_hpd_log_50":lower_hpd_log_50,
                                                   "upper_hpd_linear":upper_hpd_linear_95,"lower_hpd_linear":lower_hpd_linear_95,
                                                   "upper_hpd_linear_50":upper_hpd_linear_50, "lower_hpd_linear_50":lower_hpd_linear_50,
                                                  })
                new_df = new_df.append(local_df)
                #print(new_df)
            except:
                pass
            
    return(new_df)

In [723]:
#calculating transmission rate
def generate_summary_diff_df(input_df):
    
    
    new_df = pd.DataFrame()
   
    for i in input_df.columns.tolist():
        if "Ne." in i:
            #deme = i.split(".")[1]
           # print(deme)
            #interval = 
            #print(interval)
#             if "\n" in i.split(".")[2]:
#                 interval = i.split(".")[2][0:2]
#             else:
            interval = i.split(".")[1]
            next_interval = int(interval)+8 #averaging over two weeks to reduce noise
            local_series = input_df[i].astype('float').to_numpy()
            #print(local_series)
           
            try:
                new_df["Ne"+ ".diff." + str(interval)] = (52/8)*(input_df[i].astype("float") - input_df["Ne"+"." + str(next_interval)].astype('float'))
            
            
            except KeyError:
                pass 
            
            
    return(new_df)

In [724]:
#calculates Rt with both intros and local transmission 
def generate_local_and_intro_rt(input_df):
    
    
    new_df = pd.DataFrame()
    incubation_period = 366/8
    uninfectious_rate = 366/10.9
    
    for i in input_df.columns.tolist():
        if "Ne" in i:
            interval = i.split(".")[2]
            new_df["rt" +  "." + str(interval)] = (1+ (input_df[i].astype("float") / uninfectious_rate)) * (1+ (input_df[i].astype("float") / incubation_period))
            

            
    return(new_df)

In [725]:
# make a new dataframe that summarizes the 95% HPD estimate with mean for each deme and interval 
def generate_rt_summary_df(input_df):
    
    
    new_df = pd.DataFrame()
    count = 0
    for i in input_df.columns.tolist():
        #print(i)
        if "rt" in i:
            if count %1 == 0:
                interval = i.split(".")[1]
                #interval = i.split(".")[2]
                local_series = input_df[i].astype('float').to_numpy()
                mean_percent = local_series.mean()
                median_percent = np.median(local_series)

                hpd_95 = az.hdi(local_series, 0.95)
                lower_hpd_log_95 = hpd_95[0]
                upper_hpd_log_95 = hpd_95[1]
                hpd_50 = az.hdi(local_series, 0.50)
                lower_hpd_log_50 = hpd_50[0]
                upper_hpd_log_50 = hpd_50[1]

                try:
                    local_df = pd.DataFrame.from_dict({"interval":interval, "mean_percent":mean_percent, "median_percent":median_percent,
                                                       "upper_hpd_log_95":upper_hpd_log_95,"lower_hpd_log_95":[lower_hpd_log_95], 
                                                       "upper_hpd_log_50":upper_hpd_log_50,"lower_hpd_log_50":lower_hpd_log_50})
                    new_df = new_df.append(local_df)
                except:
                    pass
            count+=1

    return(new_df)

In [726]:
# read forward migration rates from BEAST log files
def read_in_forward_migration_rates_mascot(log_file_path):
    
    mig_rates_dict = {"sample":[]}
    
    with open(log_file_path, "r") as infile:
        line_number = 0
        for line in infile:
            #print(line_number)
            line_number += 1
            if not line.startswith("#"):  # log combiner will sometimes put the entire xml at the start of the log file
                # use the first line to find the migration rate columns
                
            # use the first line to find the migration rate columns
                if "posterior" in line:
                    all_cols = line.split("\t")
                    mig_column_indices = []   # list to store column indices
                    mig_key = {}   # dictionary to store the column index to map to column name

                    for i in range(len(all_cols)):
                        col = all_cols[i]
                        if "immigrationRate" in col: 
                            mig_column_indices.append(i)

                    # make an empty dictionary to store Nes and generate dictionary to convert index to name
                    for n in mig_column_indices:
                        name = line.split("\t")[n]
                        interval = name.split(".")[1]# the syntax here is "NeLog.state01" where 0 is deme and 1 is interval 1
                        #interval = name.split(".")[2]
                       
                        mig_key[n] = name
                        mig_rates_dict[name] = []


                # read in actual parameter estimates and store in dictionary
                else:
                    sample = line.split("\t")[0]
                    mig_rates_dict["sample"].append(sample)

                    for index in mig_column_indices:
                        name = mig_key[index]
                        mig_rates_dict[name].append(line.split("\t")[index])
                    
                
                
                
    return(mig_rates_dict)

In [727]:
# calculates the percentage of cases due to introductions. Need both the migration df and the estimates transmisssion rate from the differences in log Nes
def generate_percent_intro_df(input_df, seir_transmission_rate):
    
    temp_df = pd.DataFrame()
    new_df = pd.DataFrame()
   
    for i in input_df.columns.tolist():
        if "immigrationRate" in i:
        
            interval = i.split(".")[1]
            #deme = i.split(".")[2]

            try:
                temp_df["total."+ str(interval)] = seir_transmission_rate["Ne.diff." + str(interval)].astype("float") +  input_df[i].astype("float").apply(np.exp)
                new_df["intro.percent"+ "." + str(interval)] = input_df[i].astype("float").apply(np.exp).div(temp_df["total."+ str(interval)], axis = 0) 

            except KeyError: #this was added because not all regions have equal time periods for their epidemics and it was throwing an error everytime it had to switch deme
                pass 
                  
    return(new_df)

In [728]:
# make a new dataframe that summarizes the 95% HPD estimate with mean for each deme and interval 
def generate_summary_df_mig(input_df):
    
    
    new_df = pd.DataFrame()

    for i in input_df.columns.tolist():
        if "percent" in i:
            interval = i.split(".")[2]
            #interval = i.split(".")[3]
            local_series = input_df[i].astype('float').to_numpy()
            #print(local_series)
            mean_percent = np.median(local_series)
            hpd_95 = az.hdi(local_series, 0.95)
            lower_hpd_log_95 = hpd_95[0]
            upper_hpd_log_95 = hpd_95[1]
            hpd_50 = az.hdi(local_series, 0.50)
            lower_hpd_log_50 = hpd_50[0]
            upper_hpd_log_50 = hpd_50[1]
            

            
            
            try:
                local_df = pd.DataFrame.from_dict({ "interval":interval, "mean_percent":mean_percent, 
                                                   "upper_hpd_log_95":upper_hpd_log_95,"lower_hpd_log_95":[lower_hpd_log_95], 
                                                   "upper_hpd_log_50":upper_hpd_log_50,"lower_hpd_log_50":lower_hpd_log_50})
                new_df = new_df.append(local_df)
            except:
                pass
            
    return(new_df)


In [761]:
# now do the actual analyses
def plot_ne_and_rt_comparison(log_file_path, simulation_path):
    # read in log file for changes in Ne
    Ne_skyline = read_in_Ne_changes_mascot(log_file_path)

    # remove burn-in
    Ne_df = pd.DataFrame.from_dict(Ne_skyline)

    burnin_percent = 0.2
    print(len(Ne_df))
    rows_to_remove = int(len(Ne_df)* burnin_percent)
    Ne_df = Ne_df.iloc[rows_to_remove:]

    print(len(Ne_df))
    Ne_df = Ne_df.reset_index()


    ## calculate SEIR transmission rate using the uninfectious period and the incubation period. 
    uninfectious_rate = 366/4.5 #taken from https://www.medrxiv.org/content/10.1101/2022.08.17.22278897v1.full.pdf
    incubation_period = 366/8

    ne_diff_summary = generate_summary_diff_df(Ne_df) #summarize data in easier datatable format
    seir_transmission_rate = ((ne_diff_summary*2 + uninfectious_rate + incubation_period)**2 - (incubation_period- uninfectious_rate)**2)/(4*incubation_period)


    ####### calculate Rt from BEAST 

    rt_local_and_intro_df = generate_local_and_intro_rt(ne_diff_summary)
    summary_rt_local_and_intro_df = generate_rt_summary_df(rt_local_and_intro_df)
    summary_rt_local_and_intro_df["new_interval"] = (((summary_rt_local_and_intro_df.interval.iloc[::-1]).astype(int)))    
    columns = ["mean_percent","median_percent", "upper_hpd_log_95","lower_hpd_log_95", "upper_hpd_log_50", "lower_hpd_log_50"]
    for column in columns:
        # Create a new column for the moving average
        summary_rt_local_and_intro_df[f'{column}_MA'] = summary_rt_local_and_intro_df[column].rolling(6, min_periods =1, center = True).mean()


    ## plot BEAST Rt
    band_rt = alt.Chart(summary_rt_local_and_intro_df).mark_area(interpolate='monotone', opacity = 0.5, clip = True, color = "orange").encode(
        alt.X('new_interval', axis=alt.Axis(title="time interval")),
        alt.Y('upper_hpd_log_50_MA',axis=alt.Axis(title="Rt", grid=False)),
        alt.Y2("lower_hpd_log_50_MA")  
    ).properties(
        width=550,
        height=200)

    line_rt = alt.Chart(summary_rt_local_and_intro_df).mark_line(interpolate='monotone', opacity = 0.5, clip = True, color = "black").encode(
        alt.X('new_interval', axis=alt.Axis(title="time interval")),
        alt.Y('median_percent_MA',axis=alt.Axis(title="Rt", grid=False)),
    ).properties(
        width=550,
        height=200)

    ###### calculate Rt from simulations
    
    ## read in simulation data directly from MASTER json. 
    simulation_df = pd.read_json(simulation_path, orient = "index")
    simulation_df = simulation_df.T

    ## log transform for easier comparison with BEAST output
    simulation_df.I[simulation_df.I < 0.1] = 0.1
    simulation_df.I = simulation_df.I.astype("float")
    simulation_df["log_I"] = simulation_df.I.apply(np.log)
    simulation_df["new_time"] = simulation_df.t *366
    
    simulation_df_copy = simulation_df.copy()
    
    ## format time

    simulation_df_copy["whole_time"] = np.ceil(simulation_df_copy.t *366)
    daily_sim_cases = simulation_df_copy.groupby(["whole_time"]).agg({'IntroCount':'mean','I':'mean'}).reset_index()

    daily_sim_cases["diff_intro"] = daily_sim_cases.IntroCount.diff()
#     daily_sim_cases["diff_I"] = daily_sim_cases.I.diff() 

#     daily_sim_cases["log_I"] = daily_sim_cases.I.apply(np.log)
#     daily_sim_cases["diff"] = daily_sim_cases.I.diff()


    #define to week number
    counter = 0
    daily_sim_cases["week_time"] = np.nan
    for index, row in daily_sim_cases.iterrows():
        if row.whole_time %7 == 0:
            daily_sim_cases.loc[index,"week_time"] = counter 
            counter +=1

    #backfill
    daily_sim_cases = daily_sim_cases.bfill()

    # aggregate based on weeks
    weekly_sim_cases = daily_sim_cases.groupby(["week_time"]).agg({'diff_intro':'mean','I':'mean'}).reset_index()

    weekly_sim_cases["adj_I"] = weekly_sim_cases.I - weekly_sim_cases.diff_intro
    weekly_sim_cases["log_I"] = weekly_sim_cases.adj_I.apply(np.log)
    weekly_sim_cases["diff"] = weekly_sim_cases.log_I.diff() *(366/7)

    # estimate Rt
    incubation_period = 366/8
    uninfectious_rate = 366/4.5

    weekly_sim_cases["Rt"] = (1+ (weekly_sim_cases["diff"] / uninfectious_rate)) * (1+ (weekly_sim_cases["diff"] / incubation_period))
    weekly_sim_cases.week_time = (weekly_sim_cases.week_time - 3) #*7
    #print(weekly_sim_cases)
    #print(summary_rt_local_and_intro_df)
    weekly_sim_cases = weekly_sim_cases.iloc[4:]
    weekly_sim_cases = weekly_sim_cases.iloc[:-4]
    
    ## smooth
    columns = ["Rt"]
    for column in columns:
        # Create a new column for the moving average
        weekly_sim_cases[f'{column}_MA'] = weekly_sim_cases[column].rolling(3, min_periods =1).mean()

    summary_rt_local_and_intro_df = summary_rt_local_and_intro_df.sort_values(by = ["new_interval"])
    combinded_ne_and_sims = pd.concat([weekly_sim_cases, summary_rt_local_and_intro_df.set_index(weekly_sim_cases.index)], axis=1)

    #plot Rt from simulations
    line_rt_sim = alt.Chart(weekly_sim_cases).mark_line(interpolate='monotone', opacity = 0.8, clip = True, color = "blue").encode(
        alt.X('week_time', axis=alt.Axis(title="")),
        alt.Y('Rt_MA',axis=alt.Axis(title="Rt", grid=False), scale = alt.Scale(domain = [0.3, 3])),   
    ).properties(
        width=550,
        height=200)

    final_rt_vs_sim_plot = (band_rt + line_rt+ line_rt_sim).resolve_scale(y ="shared")
    
    
    chart = alt.Chart(combinded_ne_and_sims).mark_circle().encode(
            x = alt.X('Rt_MA', axis=alt.Axis(title="Simulation Rt")),
            y = alt.Y('median_percent_MA', axis=alt.Axis(title="Estimated Rt"))
    )

    final_rt_corr = chart + chart.transform_regression('Rt_MA','median_percent_MA').mark_line() + chart.transform_regression('Rt_MA','median_percent_MA', groupby=["sampling"], params = True).mark_text().encode(
        x=alt.value(150),  # pixels from left
        y=alt.value(50),  # pixels from top
        text='params:N',
    ).transform_calculate(
        params='"rSquared = " + round((datum.rSquared) * 100)/100 '
    ).properties(
        width=550,
        height=200)


    return final_rt_vs_sim_plot, final_rt_corr

In [762]:
simulation_path = "../simulations/simulation_results/eir_3.json"
#log_file_path = "../simulations/beast_results/simmulticoal_4_all.log"

In [763]:
folder_path = '../simulations/beast_results/sim_3_faster_clock/'

# List all files in sims folder
log_files = [(folder_path + file) for file in os.listdir(folder_path) if file.endswith('.log')]

#put everything in order
log_files.sort()

In [764]:
chart_list_ne = []
chart_list_rt = []

for file in log_files:
    base_filename = file.split("/")[3].split(".")[0]
    
    plot1, plot2 = plot_ne_and_rt_comparison(file, simulation_path)
    plot1 = plot1.properties(title="Rt values - estimated vs simulation:" + file.split("_")[-2] + "_"+ file.split("_")[-1].split(".")[0])
    plot2 = plot2.properties(title="Rt correlation- estimated vs simulation:" + file.split("_")[-2] + "_"+ file.split("_")[-1].split(".")[0])

    
    chart_list_ne.append(plot1)
    chart_list_rt.append(plot2)

5001
4001
5001
4001
26004
20804
34982
27986
48970
39176
25651
20521


In [765]:
plots_1 = alt.vconcat(*chart_list_ne)

In [766]:
plots_2 = alt.vconcat(*chart_list_rt)

In [767]:
plots_1 | plots_2

alt.HConcatChart(...)

In [768]:
def plot_percent_intro_comparison(log_file_path, simulation_path):

    ### now working on percentage of cases due to introductions
    Ne_skyline = read_in_Ne_changes_mascot(log_file_path)

    # remove burn-in
    Ne_df = pd.DataFrame.from_dict(Ne_skyline)

    burnin_percent = 0.3
    print(len(Ne_df))
    rows_to_remove = int(len(Ne_df)* burnin_percent)
    Ne_df = Ne_df.iloc[rows_to_remove:]

    print(len(Ne_df))
    Ne_df = Ne_df.reset_index()
    

    ## calculate SEIR transmission rate using the uninfectious period and the incubation period. 
    uninfectious_rate = 366/10.9 #taken from https://www.medrxiv.org/content/10.1101/2022.08.17.22278897v1.full.pdf
    incubation_period = 366/8

    ne_diff_summary = generate_summary_diff_df(Ne_df) #summarize data in easier datatable format
    seir_transmission_rate = ((ne_diff_summary*2 + uninfectious_rate + incubation_period)**2 - (incubation_period- uninfectious_rate)**2)/(4*incubation_period)

    # read in migration rate estimates from bEAST
    migration_rates_f = read_in_forward_migration_rates_mascot(log_file_path)
    mig_df_f = pd.DataFrame.from_dict(migration_rates_f)

    # remove burnin
    burnin_percent = 0.3
    #print(len(mig_df_f))
    rows_to_remove = int(len(mig_df_f)* burnin_percent)
    mig_df_f = mig_df_f.iloc[rows_to_remove:]

    #print(len(mig_df_f))
    mig_df_f = mig_df_f.reset_index()

    # calculate % of cases due to introd
    percent_df = generate_percent_intro_df(mig_df_f, seir_transmission_rate)

    # summarize data in easier to use format
    summary_percent_df = generate_summary_df_mig(percent_df)
    summary_percent_df = summary_percent_df.reset_index()

    # making sure that any numbers >1 are excluded as outliers from stochastic noise
    percent_df =pd.DataFrame(np.where(percent_df <1, percent_df, 1), columns=percent_df.columns )
    percent_df =pd.DataFrame(np.where(percent_df >0, percent_df, 0), columns=percent_df.columns )

    # format dates
    summary_percent_df["new_interval"] = (((summary_percent_df.interval.iloc[::-1]).reset_index(drop=True).astype(int))+1)   #smoothing
    summary_percent_df['days'] = (summary_percent_df.interval.astype(int)-4)*7 #the 1.5 adjustment is made due to the fact that we take the difference of Nes over three weeks 
    summary_percent_df['date'] = dt.strptime("2024-09-12",  "%Y-%m-%d") - summary_percent_df.days.map(timedelta)
    #print(summary_percent_df)
    # smoothing due to stochasitc nosie
    columns = ["mean_percent", "upper_hpd_log_95","lower_hpd_log_95", "upper_hpd_log_50", "lower_hpd_log_50"]
    for column in columns:
        # Create a new column for the moving average
        summary_percent_df[f'{column}_MA'] = summary_percent_df[column].rolling(6, min_periods =1).mean()

    #print(summary_percent_df)
    # plot % of cases due to intros from BEAST
    line1 = alt.Chart(summary_percent_df).mark_area(interpolate='monotone', opacity = 0.9, clip = True).encode(
        alt.X('new_interval', title = "Time Interval"),
        alt.Y('lower_hpd_log_50_MA',title = "% of cases due to intros", axis=alt.Axis( grid=False, format='%', ), scale=alt.Scale(domain=(0.0, 0.9))),
        alt.Y2('upper_hpd_log_50_MA' )
    ).properties(
        width=450,
        height=150)

    band = alt.Chart(summary_percent_df).mark_area(interpolate='monotone', opacity = 0.2, clip = True).encode(
        alt.X('new_interval', title = "Time Interval"),
        alt.Y('lower_hpd_log_95_MA',title = "% of cases due to intros", axis=alt.Axis( grid=False, format='%', ), scale=alt.Scale(domain=(0.0, 0.9))),
        alt.Y2('upper_hpd_log_95_MA' )
    ).properties(
        width=450,
        height=150)

    #line1
    intro_plot = (line1 + band).resolve_scale(y= "shared") 
    
    ## read in simulation dynamics
    simulation_df = pd.read_json(simulation_path, orient = "index")
    
    ## transpose for ease
    simulation_df = simulation_df.T

    ## calculate the number of new cases and introductions
    simulation_df.I = simulation_df.I.astype("float")
    simulation_df["diff_intro"] = simulation_df.IntroCount.diff()
    simulation_df["diff_I"] = simulation_df.I.diff()

    ## remove all the negatie numbers that don't represent a new intro or case
    simulation_df["diff_I"][simulation_df["diff_I"] < 0 ] = np.nan
    simulation_df["diff_intro"][simulation_df["diff_intro"] < 0 ] = np.nan
    simulation_df = simulation_df.dropna(subset=['diff_intro', 'diff_I'])

    #simulate days
    simulation_df["whole_time"] = np.floor(simulation_df.t *366)

    # aggregate to whole time due to the simulations having multiple events in the same day
    daily_sim_cases = simulation_df.groupby(["whole_time"]).agg({'diff_intro':'mean','diff_I':'mean'}).reset_index()

    # changing daily to week
    counter = 0
    daily_sim_cases["week_time"] = np.nan
    for index, row in daily_sim_cases.iterrows():
        if row.whole_time %7 == 0:
            daily_sim_cases.loc[index,"week_time"] = counter 
            counter +=1
    daily_sim_cases = daily_sim_cases.bfill()
    weekly_sim_cases = daily_sim_cases.groupby(["week_time"]).agg({'diff_intro':'mean','diff_I':'mean'}).reset_index()

    # calculate % of cases due to intros from sims
    weekly_sim_cases["percent_intro"] = weekly_sim_cases.diff_intro/(weekly_sim_cases.diff_intro+ weekly_sim_cases.diff_I)

    weekly_sim_cases.week_time = (weekly_sim_cases.week_time -1) #*7
    weekly_sim_cases = weekly_sim_cases.iloc[3:]
    weekly_sim_cases = weekly_sim_cases.iloc[:-4]
    
    #smoothing - rolling mean 
    columns = ["percent_intro"]
    for column in columns:
        # Create a new column for the moving average
        weekly_sim_cases[f'{column}_MA'] = weekly_sim_cases[column].rolling(4, min_periods =1, center = True).mean()

    
    #plot from sims
    percent_intro_sim = alt.Chart(weekly_sim_cases).mark_line(interpolate='monotone', opacity = 0.8, clip = True, color = "black").encode(
        alt.X('week_time', axis=alt.Axis(title="Time Interval")),
        alt.Y('percent_intro_MA',axis=alt.Axis(title="% of cases due to intros", grid=False)),
    ).properties(
        width=450,
        height=150)

    final_plot = line1  + percent_intro_sim
    
        
    summary_percent_df = summary_percent_df.sort_values(by = ["new_interval"])

    combinded_percent_and_sims = pd.concat([weekly_sim_cases, summary_percent_df.set_index(weekly_sim_cases.index)], axis=1)

    
    chart = alt.Chart(combinded_percent_and_sims).mark_circle().encode(
        alt.X('percent_intro_MA', title = "Truth via Simulations"),
        alt.Y('mean_percent_MA', title = "Estimated via BEAST ", scale=alt.Scale(domain=(0.0, 0.9))
    ))

    final = chart + chart.transform_regression('percent_intro_MA','mean_percent_MA').mark_line() + chart.transform_regression('percent_intro_MA','mean_percent_MA', groupby=["sampling"], params = True).mark_text().encode(
        x=alt.value(150),  # pixels from left
        y=alt.value(50),  # pixels from top
        text='params:N',
    ).transform_calculate(
        params='"rSquared = " + round((datum.rSquared) * 100)/100 ').properties(
        width=450,
        height=150)

    
    return final_plot, final

In [769]:
chart_list_percent_intro = []
corr_list_plot = []

for file in log_files:
    # Get the base filename without the extension
    base_filename = file.split("/")[3].split(".")[0]
    
    # Assign the output of plot_ne_and_rt_comparison to variables
    plot1, plot2 = plot_percent_intro_comparison(file,simulation_path )
    
    plot1 = plot1.properties(title="Estimated vs simulation:" +file.split("_")[-2] + "_"+ file.split("_")[-1].split(".")[0])
    plot2 = plot2.properties(title="Correlation - estimated vs simulation:" +file.split("_")[-2] + "_"+ file.split("_")[-1].split(".")[0])


  
    chart_list_percent_intro.append(plot1)
    corr_list_plot.append(plot2)


5001
3501
5001
3501
26004
18203
34982
24488
48970
34279
25651
17956


In [770]:
percent_sim_plots = alt.vconcat(*chart_list_percent_intro)

In [771]:
percent_sims_corr = alt.vconcat(*corr_list_plot)

In [772]:
percent_sim_plots | percent_sims_corr

alt.HConcatChart(...)

In [625]:
## read in simulation data directly from MASTER json. 
simulation_df = pd.read_json(simulation_path, orient = "index")
simulation_df = simulation_df.T

## log transform for easier comparison with BEAST output
simulation_df.I[simulation_df.I < 0.1] = 0.1
simulation_df.I = simulation_df.I.astype("float")
simulation_df["log_I"] = simulation_df.I.apply(np.log)
simulation_df["new_time"] = simulation_df.t *366

simulation_df_copy = simulation_df.copy()

## format time

simulation_df_copy["whole_time"] = np.ceil(simulation_df_copy.t *366)
daily_sim_cases = simulation_df_copy.groupby(["whole_time"]).agg({'IntroCount':'mean','I':'mean'}).reset_index()

daily_sim_cases["diff_intro"] = daily_sim_cases.IntroCount.diff()
# daily_sim_cases["diff_I"] = daily_sim_cases.I.diff() 

# daily_sim_cases["log_I"] = daily_sim_cases.I.apply(np.log)
# daily_sim_cases["diff"] = daily_sim_cases.I.diff()


#define to week number
counter = 0
daily_sim_cases["week_time"] = np.nan
for index, row in daily_sim_cases.iterrows():
    if row.whole_time %7 == 0:
        daily_sim_cases.loc[index,"week_time"] = counter 
        counter +=1

#backfill
daily_sim_cases = daily_sim_cases.bfill()

# aggregate based on weeks
weekly_sim_cases = daily_sim_cases.groupby(["week_time"]).agg({'diff_intro':'mean','I':'mean'}).reset_index()

weekly_sim_cases["adj_I"] = weekly_sim_cases.I - weekly_sim_cases.diff_intro
weekly_sim_cases["log_I"] = weekly_sim_cases.adj_I.apply(np.log)
print(weekly_sim_cases)

weekly_sim_cases["diff"] = weekly_sim_cases.log_I.diff() *(366/7)

# estimate Rt
incubation_period = 366/8
uninfectious_rate = 366/10.9

weekly_sim_cases["Rt"] = (1+ ((weekly_sim_cases["diff"]) / uninfectious_rate)) * (1+ ((weekly_sim_cases["diff"] )/ incubation_period))
weekly_sim_cases.week_time = (weekly_sim_cases.week_time -1.5) #*7
weekly_sim_cases = weekly_sim_cases.iloc[3:]

## smooth
columns = ["Rt"]
for column in columns:
    # Create a new column for the moving average
    weekly_sim_cases[f'{column}_MA'] = weekly_sim_cases[column].rolling(6, min_periods =1).mean()



#plot Rt from simulations
line_rt_sim = alt.Chart(weekly_sim_cases).mark_line(interpolate='monotone', opacity = 0.8, clip = True, color = "blue").encode(
    alt.X('week_time', axis=alt.Axis(title="")),
    alt.Y('Rt_MA',axis=alt.Axis(title="Rt", grid=False), scale = alt.Scale(domain = [0.3, 3])),   
).properties(
    width=550,
    height=200)

final_rt_vs_sim_plot = ( line_rt_sim).resolve_scale(y ="shared")
final_rt_vs_sim_plot

    week_time  diff_intro          I      adj_I     log_I
0         0.0    1.000000   0.100000  -0.900000       NaN
1         1.0    1.400000   8.526587   7.126587  1.963832
2         2.0    1.685714  14.336508  12.650794  2.537720
3         3.0    2.240816  21.234032  18.993216  2.944082
4         4.0    2.113946  31.131095  29.017150  3.367887
5         5.0    3.345238  37.673494  34.328256  3.535969
6         6.0    1.166667  32.799800  31.633133  3.454205
7         7.0    2.313853  30.419650  28.105797  3.335976
8         8.0    1.626623  23.015513  21.388889  3.062872
9         9.0    1.935714  17.948052  16.012338  2.773360
10       10.0    1.757143  19.124717  17.367574  2.854605
11       11.0    2.779365  18.981241  16.201876  2.785127
12       12.0    2.749206  17.085714  14.336508  2.662809
13       13.0    1.028571  11.286281  10.257710  2.328030
14       14.0    2.238095  11.197279   8.959184  2.192679
15       15.0    2.029762  12.426912  10.397150  2.341532
16       16.0 

alt.Chart(...)

In [632]:
## read in simulation dynamics
simulation_df = pd.read_json(simulation_path, orient = "index")
simulation_df = simulation_df.T

## log transform for easier comparison with BEAST output
#simulation_df.I[simulation_df.I < 0.1] = 0.1
simulation_df.I = simulation_df.I.astype("float")
#simulation_df["log_I"] = simulation_df.I.apply(np.log)


## now calculate the true percentage of cases due to intros from simulations

simulation_df["diff_intro"] = simulation_df.IntroCount.diff()
simulation_df["diff_I"] = simulation_df.I.diff()
simulation_df["diff_I"][simulation_df["diff_I"] < 0 ] = np.nan
simulation_df["diff_intro"][simulation_df["diff_intro"] < 0 ] = np.nan

simulation_df = simulation_df.dropna(subset=['diff_intro', 'diff_I'])

simulation_df

,R,samp2,samp,t,E,IntroCount,sim,I,Iinit,diff_intro,diff_I
1,0.0,0.0,0.0,0.000783,0.0,1.0,postSimConditions,1.0,1.0,1.0,1.0
3,0.0,0.0,0.0,0.002233,23.0,1.0,lineageEndConditions,1.0,1.0,0.0,1.0
4,0.0,0.0,0.0,0.002445,22.0,1.0,leafCountEndConditions,2.0,1.0,0.0,1.0
5,0.0,0.0,0.0,0.003921,22.0,2.0,seed,3.0,1.0,1.0,1.0
6,0.0,0.0,0.0,0.004427,21.0,2.0,model,4.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2254,398.0,0.0,565.0,0.703238,8.0,503.0,NaN,14.0,1.0,1.0,1.0
2255,398.0,0.0,565.0,0.703385,7.0,503.0,NaN,15.0,1.0,0.0,1.0
2257,399.0,0.0,565.0,0.70347,7.0,504.0,NaN,15.0,1.0,1.0,1.0
2259,399.0,0.0,566.0,0.704506,7.0,505.0,NaN,15.0,1.0,1.0,1.0


In [642]:
## read in simulation dynamics
simulation_df = pd.read_json(simulation_path, orient = "index")
simulation_df = simulation_df.T

simulation_df.I = simulation_df.I.astype("float")
simulation_df["diff_intro"] = simulation_df.IntroCount.diff()
simulation_df["diff_I"] = simulation_df.I.diff()

simulation_df["diff_I"][simulation_df["diff_I"] < 0 ] = np.nan
simulation_df["diff_intro"][simulation_df["diff_intro"] < 0 ] = np.nan

simulation_df = simulation_df.dropna(subset=['diff_intro', 'diff_I'])


simulation_df["whole_time"] = np.floor(simulation_df.t *366)

# aggregate to whole time due to the simulations having multiple events in the same day
daily_sim_cases = simulation_df.groupby(["whole_time"]).agg({'diff_intro':'mean','diff_I':'mean'}).reset_index()

# changing daily to week
counter = 0
daily_sim_cases["week_time"] = np.nan
for index, row in daily_sim_cases.iterrows():
    if row.whole_time %7 == 0:
        daily_sim_cases.loc[index,"week_time"] = counter 
        counter +=1
daily_sim_cases = daily_sim_cases.bfill()
weekly_sim_cases = daily_sim_cases.groupby(["week_time"]).agg({'diff_intro':'mean','diff_I':'mean'}).reset_index()
print(weekly_sim_cases)

# calculate % of cases due to intros from sims
weekly_sim_cases["percent_intro"] = weekly_sim_cases.diff_intro/(weekly_sim_cases.diff_intro+ weekly_sim_cases.diff_I)

weekly_sim_cases.week_time = (weekly_sim_cases.week_time) #*7
weekly_sim_cases = weekly_sim_cases.iloc[2:]
weekly_sim_cases = weekly_sim_cases.iloc[:-4]

#smoothing - rolling mean 
columns = ["percent_intro"]
for column in columns:
    # Create a new column for the moving average
    weekly_sim_cases[f'{column}_MA'] = weekly_sim_cases[column].rolling(1, min_periods =1, center = True).mean()


#plot from sims
percent_intro_sim = alt.Chart(weekly_sim_cases).mark_line(interpolate='monotone', opacity = 0.8, clip = True, color = "black").encode(
    alt.X('week_time', axis=alt.Axis(title="Time Interval")),
    alt.Y('percent_intro_MA',axis=alt.Axis(title="% of cases due to intros", grid=False)),
).properties(
    width=450,
    height=150)

final_plot = percent_intro_sim

final_plot



    week_time  diff_intro  diff_I
0         0.0    0.333333     1.0
1         1.0    0.440476     1.0
2         2.0    0.336508     1.0
3         3.0    0.246032     1.0
4         4.0    0.253808     1.0
5         5.0    0.258369     1.0
6         6.0    0.139286     1.0
7         7.0    0.309091     1.0
8         8.0    0.402381     1.0
9         9.0    0.342857     1.0
10       10.0    0.650000     1.0
11       11.0    0.650000     1.0
12       12.0    0.690476     1.0
13       13.0    0.319048     1.0
14       14.0    0.816667     1.0
15       15.0    0.468537     1.0
16       16.0    0.446032     1.0
17       17.0    0.538095     1.0
18       18.0    0.300000     1.0
19       19.0    0.660204     1.0
20       20.0    0.433333     1.0
21       21.0    0.797619     1.0
22       22.0    0.388095     1.0
23       23.0    0.676871     1.0
24       24.0    0.785714     1.0
25       25.0    0.454082     1.0
26       26.0    0.330952     1.0
27       27.0    0.457143     1.0
28       28.0 

alt.Chart(...)

In [639]:
simulation_df

,R,samp2,samp,t,E,IntroCount,sim,I,Iinit,diff_intro,diff_I,whole_time
1,0.0,0.0,0.0,0.000783,0.0,1.0,postSimConditions,1.0,1.0,1.0,1.0,1
3,0.0,0.0,0.0,0.002233,23.0,1.0,lineageEndConditions,1.0,1.0,0.0,1.0,1
4,0.0,0.0,0.0,0.002445,22.0,1.0,leafCountEndConditions,2.0,1.0,0.0,1.0,1
5,0.0,0.0,0.0,0.003921,22.0,2.0,seed,3.0,1.0,1.0,1.0,2
6,0.0,0.0,0.0,0.004427,21.0,2.0,model,4.0,1.0,0.0,1.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2254,398.0,0.0,565.0,0.703238,8.0,503.0,NaN,14.0,1.0,1.0,1.0,258
2255,398.0,0.0,565.0,0.703385,7.0,503.0,NaN,15.0,1.0,0.0,1.0,258
2257,399.0,0.0,565.0,0.70347,7.0,504.0,NaN,15.0,1.0,1.0,1.0,258
2259,399.0,0.0,566.0,0.704506,7.0,505.0,NaN,15.0,1.0,1.0,1.0,258


In [609]:
simulation_df = pd.read_json(simulation_path, orient = "index")
simulation_df

,0,1,2,3,4,5,6,7,8,9,...,2251,2252,2253,2254,2255,2256,2257,2258,2259,2260
R,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,397.000000,397.000000,398.000000,398.000000,398.000000,399.00000,399.00000,399.000000,399.000000,399.00000
samp2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.00000
samp,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,564.000000,565.000000,565.000000,565.000000,565.000000,565.00000,565.00000,566.000000,566.000000,566.00000
t,0.0,0.000783,0.001592,0.002233,0.002445,0.003921,0.004427,0.004907,0.004933,0.005211,...,0.701826,0.702221,0.703227,0.703238,0.703385,0.70347,0.70347,0.704206,0.704506,0.70492
E,0.0,0.0,24.0,23.0,22.0,22.0,21.0,20.0,19.0,19.000000,...,8.000000,8.000000,8.000000,8.000000,7.000000,7.00000,7.00000,7.000000,7.000000,7.00000
IntroCount,0.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,3.000000,...,502.000000,502.000000,502.000000,503.000000,503.000000,503.00000,504.00000,504.000000,505.000000,505.00000
sim,populationEndConditions,postSimConditions,stepper,lineageEndConditions,leafCountEndConditions,seed,model,nSamples,simulationTime,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
I,0.0,1.0,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.000000,...,15.000000,14.000000,13.000000,14.000000,15.000000,14.00000,15.00000,14.000000,15.000000,15.00000
Iinit,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1.00000


In [622]:
sim_df = pd.read_json(simulation_path, orient = "index").T

# pull out new introductions
new_introductions = pd.DataFrame({'t': sim_df['t'][sim_df['IntroCount'].diff() > 0], 'intro': 1}) # highlight these times as having an intro by intro = 1

# pull out new infs
new_infected = pd.DataFrame({ 't': sim_df['t'][sim_df['I'].diff() > 0], 'val': sim_df['I'].diff()[sim_df['I'].diff() > 0]})

# merge so that both infections and introductions are in a concatinated df. The time when there are infections but no intros are identified as intro = 0 
new_introductions = pd.concat([new_introductions, pd.DataFrame({'t': new_infected['t'].repeat(new_infected['val'].astype(int)), 'intro': 0})])

introductions = new_introductions.reset_index(drop=True)

In [623]:
introductions

,t,intro
0,0.000783,1
1,0.003921,1
2,0.005211,1
3,0.007481,1
4,0.009935,1
...,...,...
1637,0.701826,0
1638,0.703238,0
1639,0.703385,0
1640,0.70347,0
